# P-027 — E7 real-data evaluation (CIFAR-10, Colab T4)

The external audit of **[The Dissipation Budget Law](https://srivtx.github.io/p-rick/paper/p027.html)** settled the
synthetic question (E6, 96 runs: the port-Hamiltonian stream
trains to parity at depth 240 with no normalizer). The standing
objection is scale: *nothing there competes on real data.*
This notebook runs that test, in the order the audit demanded:

| stage | what | status at notebook creation |
|---|---|---|
| 0 | **math first** — the 13-assertion law suite runs before any training | laws are theorems, checked independently of benchmarks |
| 1 | **E6b** — the trunk-LayerNorm ResNet ablation (the one missing baseline) | specified in paper §4.6, **not yet run** — this notebook runs it |
| 2 | **E7a** — CIFAR-10, 5 architectures × depths {12, 48, 96} × 3 seeds × 2 LRs | pre-registered, **not yet run** |
| 3 | full metric ledger: losses/accuracy, per-layer activation + gradient profiles, NaN/divergence events, params, wall-clock, throughput, peak memory | recorded per run |
| 4 | exact commands, seeds, splits, versions, device recorded in the results JSON | machine-readable |
| 5 | decision rules evaluated **mechanically** from the pre-registration | `figures/p-027/e7-preregistration.json`, written before any run |

**Pre-registered decision rules (the short version).** Evidence
*for* requires ALL of: PH accuracy within 1.0 pt of the best
baseline at every depth; lowest forward growth at L ≥ 48 with
zero port-budget violations; zero PH failed runs; efficiency
within 1.5× on time and memory. Evidence *against*: any accuracy
deficit > 1.5 pt, any PH NaN/divergence, or no norm-signature
difference. A tie with no instability events anywhere is reported
as the neutral outcome it is.

**GPU is required.** The notebook hard-fails without CUDA — no
silent CPU fallback. (All 96 original E6 runs landed on CPU
because a micro-benchmark honestly preferred CPU for tiny d=32
nets; that made every wall-clock column a CPU number. Fixed
here.)

**How to use this notebook**
1. `Runtime -> Change runtime type -> T4 GPU -> Save`
2. `Runtime -> Run all` (Ctrl+F9)
3. The benchmark cell prints the **projected total time** before
   the long cell starts — expect several hours for the full grid;
   every completed run saves, and re-running resumes exactly
   where it stopped (safe to interrupt, safe to split across
   sessions)
4. The last cell prints the results JSON **between two marker
   lines** and auto-downloads it. Paste the payload (or upload
   the file) back to the assistant, who writes the numbers into
   the paper, regenerates Figure 8, and ships — honestly,
   including a negative result if that is what the data says.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'NO GPU RUNTIME. E7 requires CUDA (wall-clock, throughput and',
    'peak-memory comparisons are GPU-only). Fix it:',
    'Runtime -> Change runtime type -> T4 GPU -> Save, then',
    'Runtime -> Run all again.')
print('GPU      :', torch.cuda.get_device_name(0))
print('VRAM     : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 2**30))
print('torch    :', torch.__version__)
print('cudnn    :', torch.backends.cudnn.version())
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

In [ ]:
import json, os, urllib.request

os.makedirs('p-rick/code', exist_ok=True)
os.makedirs('p-rick/figures/p-027', exist_ok=True)
BASE = 'https://raw.githubusercontent.com/srivtx/p-rick/main/'
FILES = ['code/p-027b-training.py',
         'code/p-027c-cifar.py',
         'code/p-027d-cifar-figures.py',
         'code/p-027-tests.py',
         'figures/p-027/e7-preregistration.json',
         'figures/p-027/training-results.json']
for f in FILES:
    dst = os.path.join('p-rick', f)
    urllib.request.urlretrieve(BASE + f, dst)
    print('fetched', f)

pre = json.load(open('p-rick/figures/p-027/e7-preregistration.json'))
e6 = json.load(open('p-rick/figures/p-027/training-results.json'))
print()
print('pre-registration: registered %s, before any E7 run: %s'
      % (pre['registered_at'], pre['registered_before_any_e7_run']))
print('E6 state on file: %d runs, complete=%s, models=%s'
      % (len(e6['runs']), e6.get('complete'),
         sorted(set(r['model'] for r in e6['runs']))))
if not os.path.exists('p-rick/figures/p-027/cifar-results.json'):
    print('E7 state on file: none — fresh start (90 runs)')

## Stage 0 — the mathematics, checked before any benchmark

The audit asked for the bound to be verified **separately** from
empirical performance. `code/p-027-tests.py` holds the 13-assertion
law suite: Cayley round-trips, the damping-spectrum conditionality
(including the audit's own R = 4I counterexample), the conservation
law against the exact accumulated bound, the volume law, and the
budget identities. Training numbers can say whatever they like;
these must pass regardless. (Additionally, every trained PH run in
E7 self-checks the port budget on real batches and records the
violation count — expected zero.)

In [ ]:
!python p-rick/code/p-027-tests.py

## Stage 1 — E6b: the trunk-LayerNorm ablation (24 runs)

The one baseline missing from the 96-run E6 sweep: a ResNet whose
**trunk** is LayerNormed every layer (`z' = LN(z + g(z))`) — the
placement that actually pins trunk scale, at the price of making
that scale a per-sample statistic rather than an operator bound.
This is the comparison §4.6 of the paper specifies and the
explainer page registers. **Registered prediction:** forward growth
O(1) (the trunk is re-scaled by construction), accuracy at parity
with the branch-LN ResNet, and no operator bound — which is the
point. Same protocol as the original 96 runs; results land in the
same `training-results.json`, purely additive. ~30–60 min on T4.

In [ ]:
!python p-rick/code/p-027b-training.py --device=cuda --models=resnet-tln

In [ ]:
import json, statistics as st
from collections import defaultdict

doc = json.load(open('p-rick/figures/p-027/training-results.json'))
runs = doc['runs']
by = defaultdict(dict)
for r in runs:
    k = (r['task'], r['depth'], r['model'])
    if r['seed'] not in by[k] or r['val_acc'] > by[k][r['seed']]['val_acc']:
        by[k][r['seed']] = r

print(f"{'task':9s} {'depth':>5s} {'model':11s} {'test acc':>16s} "
      f"{'fwd growth':>12s} {'grad max':>9s}")
for (task, depth, model), seeds in sorted(by.items()):
    accs = [s['test_acc'] for s in seeds.values()]
    fg = [s['fwd_growth'] for s in seeds.values()]
    gm = [s['grad_max_ratio'] for s in seeds.values()]
    print(f"{task:9s} {depth:5d} {model:11s} "
          f"{st.mean(accs):8.3f} +- {st.stdev(accs) if len(accs)>1 else 0:.3f} "
          f"{st.mean(fg):9.1f}x {st.mean(gm):9.2f}")
print()
print('total runs:', len(runs), '| complete:', doc.get('complete'))
print('devices:', sorted(set(r['device'] for r in runs)))

## Stage 2 — benchmark and time projection (no results written)

Before the sweep, one warm measurement per (model, depth): a few
training steps at batch 256 on the real input shape. The cell
prints the **projected total** for the full E7a grid (90 runs,
upper bound — early stopping usually cuts it). If the projection
exceeds your session, run the sweep with a smaller `BUDGET_S`
anyway: the harness stops cleanly and re-running resumes.

In [ ]:
import importlib.util, time, torch
import torch.nn.functional as F

spec = importlib.util.spec_from_file_location('h', 'p-rick/code/p-027c-cifar.py')
h = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h)
torch.backends.cudnn.benchmark = True

def bench(model_name, depth, steps=8, warmup=2):
    torch.manual_seed(0)
    m = h.StreamNet(model_name, depth, n_classes=10).to('cuda')
    opt = torch.optim.Adam(m.parameters(), lr=3e-4)
    x = torch.randn(256, 3, 32, 32, device='cuda')
    y = torch.randint(0, 10, (256,), device='cuda')
    for _ in range(warmup):
        loss = F.cross_entropy(m(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(steps):
        loss = F.cross_entropy(m(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    torch.cuda.synchronize()
    del m, opt
    torch.cuda.empty_cache()
    return (time.time() - t0) / steps

STEPS_PER_EPOCH = 176   # ceil(45000 / 256)
MODELS = ['ph', 'resnet', 'resnet-ln', 'resnet-tln', 'cayley']
DEPTHS = [12, 48, 96]
per_epoch = {}
for m in MODELS:
    for d in DEPTHS:
        s = bench(m, d)
        per_epoch[(m, d)] = s * STEPS_PER_EPOCH
        print(f'{m:11s} depth {d:3d}: {s*1000:6.1f} ms/step  ->  {per_epoch[(m, d)]:6.1f} s/epoch')

total = 30 * 3 * 2 * sum(per_epoch.values())   # epochs x seeds x LRs
print(f'\nprojected E7a total (upper bound, no early stopping): {total/3600:.1f} h')
print('(each run saves; interrupt and re-run any time — the sweep resumes)')

## The E7a sweep — the long cell

5 architectures × depths {12, 48, 96} × 3 seeds × 2 learning rates,
identical protocol, learning rate selected on validation, test set
touched once per run. Every job prints a line as it finishes. Set
`BUDGET_S` below to cap the cell (seconds; `None` = run to
completion). Progress saves after every single run.

In [ ]:
BUDGET_S = None    # e.g. 5400 to stop after 90 min and resume later
cmd = 'python p-rick/code/p-027c-cifar.py --device=cuda'
if BUDGET_S:
    cmd += f' --budget={BUDGET_S}'
print(cmd)
!{cmd}

In [ ]:
from IPython.display import Image, display
import os

f8 = 'p-rick/figures/p-027/f8-cifar.png'
if os.path.exists(f8):
    display(Image(f8))
else:
    print('f8-cifar.png not written yet - the sweep was interrupted;')
    print('re-run the notebook to finish the remaining jobs.')

## Stage 5 — the digest, evaluated against the pre-registration

The rules below are not improvised after seeing the numbers — they
are the ones written into `e7-preregistration.json` before any run
landed. The cell applies them mechanically and prints PASS / FAIL /
PENDING per criterion. The human write-up happens after, and a
negative result is reported, not buried.

In [ ]:
import json, statistics as st
from collections import defaultdict

doc = json.load(open('p-rick/figures/p-027/cifar-results.json'))
runs = [r for r in doc['runs'] if r['dataset'] == 'cifar10']
by = defaultdict(dict)
for r in runs:
    k = (r['model'], r['depth'])
    if r['seed'] not in by[k] or r['best_val_acc'] > by[k][r['seed']]['best_val_acc']:
        by[k][r['seed']] = r

def agg(m, d, key):
    pick = by.get((m, d), {})
    return [r[key] for r in pick.values()] or []

DEPTHS = sorted(set(d for (_, d) in by))
print(f"{'model':11s} {'depth':>5s} {'test acc':>16s} {'growth':>8s} "
      f"{'gmax':>6s} {'s/ep':>6s} {'MB':>6s} {'params':>8s} {'div':>4s} {'nan':>4s} {'viol':>5s}")
for m in ['ph', 'resnet', 'resnet-ln', 'resnet-tln', 'cayley']:
    for d in DEPTHS:
        accs, gr = agg(m, d, 'test_acc'), agg(m, d, 'act_growth')
        if not accs:
            continue
        div = sum(1 for r in by[(m, d)].values() if r['diverged'])
        nan = sum(r['nan_events'] for r in by[(m, d)].values())
        viol = sum((r['port_violations'] or 0) for r in by[(m, d)].values())
        spe = st.mean([r['sec_per_epoch'] for r in by[(m, d)].values()])
        mem = st.mean([r['peak_mem_mb'] or 0 for r in by[(m, d)].values()])
        par = by[(m, d)][0]['params']
        print(f"{m:11s} {d:5d} {st.mean(accs):8.3f} +- {st.stdev(accs) if len(accs)>1 else 0:.3f} "
              f"{st.mean(gr):6.1f}x {st.mean(agg(m, d, 'grad_max_ratio')):6.2f} "
              f"{spe:6.1f} {mem:6.0f} {par/1e6:7.2f}M {div:4d} {nan:4d} {viol:5d}")

print(f'\nruns on file: {len(runs)} / 90   complete: {doc.get("complete")}')
print('devices:', sorted(set(r["device"] for r in runs)))

# ---- mechanical application of the pre-registered decision rules ----
def best_baseline_acc(d):
    return max((st.mean(agg(m, d, 'test_acc')) for m in
                ['resnet', 'resnet-ln', 'resnet-tln', 'cayley']
                if agg(m, d, 'test_acc')), default=float('nan'))

checks = []
for d in DEPTHS:
    ph = st.mean(agg('ph', d, 'test_acc'))
    checks.append((f'non-inferior accuracy @L={d} (>= best - 1.0pt)',
                   'PASS' if ph >= best_baseline_acc(d) - 0.01 else 'FAIL'))
for d in [x for x in DEPTHS if x >= 48]:
    ph_g = st.mean(agg('ph', d, 'act_growth'))
    rs_g = st.mean(agg('resnet', d, 'act_growth'))
    ln_g = st.mean(agg('resnet-ln', d, 'act_growth'))
    checks.append((f'lowest forward growth @L={d}',
                   'PASS' if ph_g < rs_g and ph_g < ln_g else 'FAIL'))
viol = sum((r['port_violations'] or 0) for r in runs if r['model'] == 'ph')
checks.append(('port-budget violations = 0 (PH runs)',
               'PASS' if viol == 0 else 'FAIL'))
ph_div = sum(1 for r in runs if r['model'] == 'ph' and r['diverged'])
base_div = sum(1 for r in runs if r['model'] != 'ph' and r['diverged'])
checks.append(('zero PH divergent runs', 'PASS' if ph_div == 0 else 'FAIL'))
checks.append(('instability ledger (PH vs baselines)',
               f'PH {ph_div} diverged vs baselines {base_div}'))
if 96 in DEPTHS:
    fast = min(st.mean(agg(m, 96, 'sec_per_epoch')) for m in
               ['resnet', 'resnet-ln', 'resnet-tln', 'cayley']
               if agg(m, 96, 'test_acc'))
    ph_t = st.mean(agg('ph', 96, 'sec_per_epoch'))
    checks.append((f'efficiency guard @L=96 (<= 1.5x fastest, now {ph_t/fast:.2f}x)',
                   'PASS' if ph_t <= 1.5 * fast else 'FAIL'))

print('\n--- pre-registered decision rules (mechanical) ---')
for name, res in checks:
    print(f'{res:>8s}   {name}')
pending = len(runs) < 90
note = ('grid incomplete - criteria above are provisional'
        if pending else 'grid complete')
print('\nNOTE:', note)

In [ ]:
import json, os
from google.colab import files

RES = 'p-rick/figures/p-027/cifar-results.json'
doc = json.load(open(RES))
txt = open(RES).read()
print(f"cifar-results.json: {len(txt)/1000:.1f} KB, "
      f"{len(doc['runs'])} runs, complete={doc.get('complete')}")
print()
print('=' * 25 + ' COPY EVERYTHING BELOW THIS LINE ' + '=' * 25)
print(txt)
print('=' * 25 + ' COPY EVERYTHING ABOVE THIS LINE ' + '=' * 25)
files.download(RES)
f8 = 'p-rick/figures/p-027/f8-cifar.png'
if os.path.exists(f8):
    files.download(f8)
e6 = 'p-rick/figures/p-027/training-results.json'
if os.path.exists(e6):
    files.download(e6)

## Then what

Paste the payload between the marker lines back to the assistant
(or upload the downloaded files). The assistant will:

1. write the E6b trunk-LN numbers into paper §4.6/§6 and the
   LayerNorm explainer — with the registered prediction checked
   against what actually happened,
2. write the E7 CIFAR-10 numbers into §6 and regenerate Figure 8,
3. apply the pre-registered decision rules in the paper's own
   voice — **including a negative result if that is what the data
   says**, with an analysis of why,
4. rebuild the typeset PDF, update the site cards, README, and
   push.

**E8 (CIFAR-100) is gated:** run it only after E7a has landed and
been written into the paper — set `RUN_E8 = True` below then
re-run everything (it reuses the same resumable harness).

In [ ]:
RUN_E8 = False   # gated: only after E7a is landed and written up
if RUN_E8:
    !python p-rick/code/p-027c-cifar.py --device=cuda --dataset=cifar100 --depths=48,96
else:
    print('E8 (CIFAR-100) gated off - see the note above.')